In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import optuna
import sys
from pathlib import Path
import torch # Torch needs to be installed manually if you want cuda. See installation command on pytorch website

DIR_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(DIR_ROOT))

DEVICE = "GPU" if torch.cuda.is_available() else "CPU"

%load_ext autoreload
%autoreload 2

/home/koyo-ai/miniconda3/envs/homeless/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 987
TRACK = "off_market"                     # "off_market" or "list_aware"
VAL_START = pd.Timestamp("2025-06-01")    # sales on or after this date are validation
N_TRIALS = 10                            # optuna trials
N_ESTIMATORS = 2000                      # max boosting rounds per trial
EARLY_STOPPING_ROUNDS = 100
K_FOLDS = 5                              # time series cross validation folds
DEBUG = True                             # print MdAPE and MAPE for every fold

In [3]:
import data

listings = data.load("listings")
off_market = data.load("off_market")
list_aware = data.load("list_aware")

In [4]:
from cooler_features import off_market_features, list_aware_features

In [5]:
make_features = off_market_features if TRACK == "off_market" else list_aware_features

train = listings[listings["date_sold"] < VAL_START]
val = listings[listings["date_sold"] >= VAL_START]

X_train = make_features(train)
X_val = make_features(val)

y_train = np.log(train["sold_price"])
y_val = np.log(val["sold_price"])

CAT_FEATURES = X_train.select_dtypes("category").columns.tolist()

print(f"train: {len(train):,} sales up to {train['date_sold'].max().date()}")
print(f"val:   {len(val):,} sales from {val['date_sold'].min().date()}")

train: 127,762 sales up to 2025-05-31
val:   22,238 sales from 2025-06-01


In [6]:
# TODO: Try different losses
def mdape(y_log, pred_log):
    return 100 * np.median(np.abs(np.exp(pred_log) / np.exp(y_log) - 1))

loss = mdape

In [7]:
from sklearn.model_selection import TimeSeriesSplit


def cross_val(X, y, params, K=5, debug=False, validation_loss=mdape):
    """Walk-forward K-fold: each fold trains on every row before its validation block.
    X and y must be sorted by time (listings is sorted by date_sold).
    Returns the mean validation loss over the folds."""
    scores = [] 
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=K).split(X), start=1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostRegressor(**params, task_type=DEVICE, iterations=N_ESTIMATORS,
                                  early_stopping_rounds=EARLY_STOPPING_ROUNDS, random_seed=SEED,
                                  cat_features=CAT_FEATURES, verbose=False)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
        pred = model.predict(X_va)
        scores.append(validation_loss(y_va, pred))

        if debug:
            error = 100 * np.abs(np.exp(pred) / np.exp(y_va) - 1)
            print(f"fold {fold}: train {len(train_idx):,}  val {len(val_idx):,}  "
                  f"MdAPE {error.median():.2f}%  MAPE {error.mean():.2f}%", flush=True)

    return np.mean(scores)

In [8]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, step=0.01),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0, step=0.1),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bootstrap_type": "Bayesian",
    }
    score = cross_val(X_train, y_train, params, K=K_FOLDS, debug=DEBUG, validation_loss=loss)
    return score

In [9]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED), study_name="VAPE GOD VAPE GOD VAPE GOD 420 420 696969!!!!")
study.optimize(objective, n_trials=N_TRIALS)

print("best val MdAPE:", round(study.best_value, 2))
print("best params:", study.best_params)

[I 2026-10-10 19:37:44,238] A new study created in memory with name: VAPE GOD VAPE GOD VAPE GOD 420 420 696969!!!!


fold 1: train 21,297  val 21,293  MdAPE 11.16%  MAPE 13.83%
fold 2: train 42,590  val 21,293  MdAPE 11.20%  MAPE 14.04%
fold 3: train 63,883  val 21,293  MdAPE 10.64%  MAPE 13.16%
fold 4: train 85,176  val 21,293  MdAPE 10.48%  MAPE 13.52%
fold 5: train 106,469  val 21,293  MdAPE 10.45%  MAPE 13.26%


[I 2026-10-10 19:39:14,238] Trial 0 finished with value: 10.78562415631232 and parameters: {'learning_rate': 0.05, 'depth': 6, 'l2_leaf_reg': 3.276906511377122, 'random_strength': 0.06863045404697059, 'bagging_temperature': 0.9, 'border_count': 237}. Best is trial 0 with value: 10.78562415631232.


fold 1: train 21,297  val 21,293  MdAPE 11.22%  MAPE 13.64%
fold 2: train 42,590  val 21,293  MdAPE 11.05%  MAPE 13.59%
fold 3: train 63,883  val 21,293  MdAPE 10.49%  MAPE 13.08%
fold 4: train 85,176  val 21,293  MdAPE 10.06%  MAPE 13.09%
fold 5: train 106,469  val 21,293  MdAPE 10.29%  MAPE 13.16%


[I 2026-10-10 19:41:15,973] Trial 1 finished with value: 10.620388084087981 and parameters: {'learning_rate': 0.09999999999999999, 'depth': 10, 'l2_leaf_reg': 0.0013187949354214168, 'random_strength': 0.002296326109404726, 'bagging_temperature': 0.9, 'border_count': 46}. Best is trial 1 with value: 10.620388084087981.


...

fold 1: train 21,297  val 21,293  MdAPE 11.44%  MAPE 13.91%
fold 2: train 42,590  val 21,293  MdAPE 11.78%  MAPE 14.39%
fold 3: train 63,883  val 21,293  MdAPE 11.11%  MAPE 13.63%
fold 4: train 85,176  val 21,293  MdAPE 10.94%  MAPE 13.90%
fold 5: train 106,469  val 21,293  MdAPE 11.12%  MAPE 14.08%


[I 2026-10-10 19:54:40,569] Trial 8 finished with value: 11.275664989304412 and parameters: {'learning_rate': 0.03, 'depth': 5, 'l2_leaf_reg': 1.0459159697911138, 'random_strength': 0.1157967959831502, 'bagging_temperature': 0.7000000000000001, 'border_count': 105}. Best is trial 2 with value: 10.20485530094714.


fold 1: train 21,297  val 21,293  MdAPE 11.53%  MAPE 14.17%
fold 2: train 42,590  val 21,293  MdAPE 12.01%  MAPE 14.45%
fold 3: train 63,883  val 21,293  MdAPE 10.11%  MAPE 12.59%
fold 4: train 85,176  val 21,293  MdAPE 9.88%  MAPE 12.90%
fold 5: train 106,469  val 21,293  MdAPE 9.47%  MAPE 12.36%


[I 2026-10-10 19:55:43,578] Trial 9 finished with value: 10.598877159022038 and parameters: {'learning_rate': 0.08, 'depth': 8, 'l2_leaf_reg': 0.0011955692818720264, 'random_strength': 0.022743159590986432, 'bagging_temperature': 0.2, 'border_count': 58}. Best is trial 2 with value: 10.20485530094714.


best val MdAPE: 10.2
best params: {'learning_rate': 0.08, 'depth': 9, 'l2_leaf_reg': 0.017067100856910322, 'random_strength': 0.01042489610529085, 'bagging_temperature': 0.8, 'border_count': 238}


In [10]:
from optuna.importance import get_param_importances

importance = get_param_importances(study)

for param, score in importance.items():
    print(f"{param}: {score:.4f}")

border_count: 0.3875
l2_leaf_reg: 0.3053
random_strength: 0.2035
learning_rate: 0.0406
bagging_temperature: 0.0367
depth: 0.0264


In [11]:
model = CatBoostRegressor(**study.best_params,
                          bootstrap_type="Bayesian",
                          cat_features=CAT_FEATURES,
                          task_type=DEVICE,
                          early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                          iterations=N_ESTIMATORS,
                          random_seed=SEED,
                          verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val))

predicted = np.exp(model.predict(X_val))
error = 100 * np.abs(predicted / val["sold_price"] - 1)
print(f"{TRACK}  MdAPE {error.median():.2f}%  MAPE {error.mean():.2f}%")

off_market  MdAPE 10.17%  MAPE 13.38%


In [12]:
feature_importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
display(feature_importances)

community      28.476553
square_feet    17.634043
longitude       9.475637
latitude        9.167312
house_type      8.862716
washrooms       6.446196
lot_front       6.366413
months          4.359474
lot_depth       3.146188
parking         2.695827
season          1.310357
house_style     1.200667
bedrooms        0.858618
dtype: float64